In [1]:
import numpy as np
import psutil
import subprocess
import random as rd
import pickle
import gc
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import LogarithmicMapper

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

In [2]:
n_x = 4
n_y = 1
n_site = n_x * n_y
n_qubit = n_site
dim = 2**n_qubit

n_qubit_circuit = n_qubit + 1

n_dimer = n_site//2

core  = 0
cores = 1

In [ ]:
from qiskit import QuantumRegister, QuantumCircuit
qr_circuit = QuantumRegister(n_qubit_circuit,name='q')
#qr = qr_circuit[1:]
#qm = qr_circuit[0]
#indx_qm = 0
indx_qm = 2
qm = qr_circuit[indx_qm]
qr = qr_circuit[:indx_qm] + qr_circuit[indx_qm+1:]
#print(qm,qr)

In [4]:
J                   = 1.0
Delta               = -1.0
h                   = 0.0

In [5]:
n_inter             = 2
t_inter_max         = 1.0
t_inters            = [0.0, 1.0]
hamiltonians        = []
mapper = LogarithmicMapper()
for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    hamiltonian_list = []
    # intra-dimer terms
    for i in range(0,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*Delta/4)
        hamiltonian_list.append(term)
        term = ('Z',[ii],-h/2)
        hamiltonian_list.append(term)
        term = ('Z',[jj],-h/2)
        hamiltonian_list.append(term)
    # inter-dimer terms
    for i in range(1,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*t_inter*Delta/4)
        hamiltonian_list.append(term)
    print(hamiltonian_list)
    hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonians.append(hamiltonian.simplify())

    if (core==0):
        print(t_inter, single_line(str(hamiltonians[i_inter])))
        print('')

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [1, 2], -0.0), ('YY', [1, 2], -0.0), ('ZZ', [1, 2], 0.0)]
0.0 SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [1, 2], -0.25), ('YY', [1, 2], -0.25), ('ZZ', [1, 2], 0.25)]
1.0 SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII', 'IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])



In [ ]:
init_circuit = QuantumCircuit(qr_circuit)
# qm is at the middle of 1 and 2 for 4 site
n_dimer_half = n_dimer//2
index_attached_to_qm = n_qubit//2-1

#  geometry ; qm -- qr[n_qubit//2-1] -- ... -- qr[1] -- qr[0]
#                   \--qr[n_qubit//2] -- ... -- qr[n_qubit-1]
init_circuit.cx(qm,qr[index_attached_to_qm])

init_circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
for i_dimer in range(1,n_dimer_half+1):
    i_qubit = 2*(i_dimer+n_dimer_half-1)
    init_circuit.s(qr[i_qubit+1])
    init_circuit.h(qr[i_qubit+1])
    init_circuit.t(qr[i_qubit+1])
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])
    init_circuit.tdg(qr[i_qubit+1])
    if (i_dimer<n_dimer_half):
        init_circuit.sx(qr[i_qubit+1])
        init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])
        init_circuit.x(qr[i_qubit+1])
        init_circuit.h(qr[i_qubit+1])
    else:
        init_circuit.h(qr[i_qubit+1])
        init_circuit.sdg(qr[i_qubit+1])
    init_circuit.cx(qr[i_qubit+1],qr[i_qubit])

for i_dimer in range(n_dimer_half,0,-1):
    i_qubit = 2*i_dimer-1
    init_circuit.s(qr[i_qubit-1])
    init_circuit.h(qr[i_qubit-1])
    init_circuit.t(qr[i_qubit-1])
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
    init_circuit.tdg(qr[i_qubit-1])
    if (i_dimer>1):
        init_circuit.sx(qr[i_qubit-1])
        init_circuit.cx(qr[i_qubit-1],qr[i_qubit-2])
        init_circuit.x(qr[i_qubit-1])
        init_circuit.h(qr[i_qubit-1])
    else:
        init_circuit.h(qr[i_qubit-1])
        init_circuit.sdg(qr[i_qubit-1])
    init_circuit.cx(qr[i_qubit-1],qr[i_qubit])



print(init_circuit)
init_circuit_inv = init_circuit.inverse()
print(init_circuit_inv)
# This circuit is true only for 4 qubits

                                                          
q_0: ──■──────────────────────────────────────────────────
       │  ┌───┐┌───┐┌───┐ ┌───┐ ┌─────┐ ┌───┐ ┌─────┐     
q_1: ──┼──┤ S ├┤ H ├┤ T ├─┤ X ├─┤ Tdg ├─┤ H ├─┤ Sdg ├──■──
     ┌─┴─┐└───┘└───┘└───┘ └─┬─┘ └─────┘ └───┘ └─────┘┌─┴─┐
q_2: ┤ X ├──■───────────────■────────────────────────┤ X ├
     └───┘┌─┴─┐                                ┌───┐ └───┘
q_3: ─────┤ X ├───────■────────────────────────┤ X ├──────
     ┌───┐├───┤┌───┐┌─┴─┐┌─────┐ ┌───┐ ┌─────┐ └─┬─┘      
q_4: ┤ S ├┤ H ├┤ T ├┤ X ├┤ Tdg ├─┤ H ├─┤ Sdg ├───■────────
     └───┘└───┘└───┘└───┘└─────┘ └───┘ └─────┘            
                                                        
q_0: ────────────────────────────────────────■──────────
          ┌───┐┌───┐┌───┐┌───┐┌─────┐┌───┐   │   ┌─────┐
q_1: ──■──┤ S ├┤ H ├┤ T ├┤ X ├┤ Tdg ├┤ H ├───┼───┤ Sdg ├
     ┌─┴─┐└───┘└───┘└───┘└─┬─┘└─────┘└───┘ ┌─┴─┐ └─────┘
q_2: ┤ X ├─────────────────■─────■─────────┤ X ├────────
     ├───

In [7]:
n_hamiltonians = len(hamiltonians)
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))

if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

if (core==0):
    print('# Reduced Hamiltonian differences_list')
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_list = []
    factor_XX = 2 # 1 for XX, 1 for YY
    ii = index_attached_to_qm
    jj = index_attached_to_qm+1
    term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*factor_XX)
    hamiltonian_list.append(term)
    factor_ZZ = 1
    term = ('ZZ',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])*Delta/4*factor_ZZ)
    hamiltonian_list.append(term)
    print(hamiltonian_list)
    dH = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonian_diffs_reduced.append(dH.simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

# Hamiltonian differences
0 SparsePauliOp(['IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j])
# Hamiltonian differences_list
[('IXXI', (-0.25+0j)), ('IYYI', (-0.25+0j)), ('IZZI', (0.25+0j))]
# Reduced Hamiltonian differences_list
[('XX', [1, 2], -0.5), ('ZZ', [1, 2], 0.25)]
0 SparsePauliOp(['IXXI', 'IZZI'],              coeffs=[-0.5 +0.j,  0.25+0.j])
# Hamiltonian differences_reduced_list
[('IXXI', (-0.5+0j)), ('IZZI', (0.25+0j))]


In [8]:
#init_circuit = QuantumCircuit(qr_circuit)
## qm is at the middle of 1 and 2 for 4 site
#indx_before_qm = 1 # this index is in qr
#indx_after_qm  = 2 # this index is in qr
#
## qm is at the middle of 3 and 4 for 8 site
##indx_before_qm = 3 # this index is in qr
##indx_after_qm  = 4 # this index is in qr
#
## controlled CNOTs to give initial computational states
#init_circuit.cx(qm,qr[indx_before_qm])
#init_circuit.cx(qm,qr[indx_after_qm])
#for i_qubit in range(indx_before_qm,1,-2):
#    init_circuit.cx(qr[i_qubit],qr[i_qubit-2])
#
#for i_qubit in range(indx_after_qm, n_qubit-2, +2):
#    init_circuit.cx(qr[i_qubit],qr[i_qubit+2])
#
## transformation to the bell states
## before qm
#for i_qubit in range(indx_before_qm,0,-2):
#    ii = i_qubit
#    jj = i_qubit-1
#    init_circuit.h(qr[jj])
#    init_circuit.cx(qr[jj],qr[ii])
## after qm
#for i_qubit in range(indx_after_qm, n_qubit-1, +2):
#    ii = i_qubit
#    jj = i_qubit+1
#    init_circuit.h(qr[jj])
#    init_circuit.cx(qr[jj],qr[ii])
#print(init_circuit)
#init_circuit_inv = init_circuit.inverse()
#print(init_circuit_inv)

In [ ]:
#from qiskit.quantum_info import Operator
#unitary = Operator(init_circuit).data
#vec_0 = np.zeros((dim*2),dtype=complex)
#vec_0[0] = 1
#vec_0 = unitary@vec_0
#
#vec_1 = np.zeros((dim*2),dtype=complex)
#vec_1[1] = 1
#vec_1 = unitary@vec_1
#print(vec_0[0::2])
#print(vec_1[1::2])
#print(vec_0[0::2].conj()@hamiltonians[0].to_matrix()@vec_0[0::2])
#print(vec_1[1::2].conj()@hamiltonians[0].to_matrix()@vec_1[1::2])
#
#
## checking validity of init matrix -> checked

[ 1.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j  0.00000000e+00-5.55111512e-17j
  0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00-5.55111512e-17j  0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j -3.08148791e-33+0.00000000e+00j]
[0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0.5+0.j 0. +0.j 0. +0.j
 0.5+0.j 0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
(0.49999999999999956+0j)
(-1.4999999999999987+0j)


In [76]:
nmc = 300
beta = 2.0
n_shot = 4000

n_obs = 3
# 0; norm, 1; dE1, 2; dE2
O_timelists         = [[[[] for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]

exact_dir = '../exact'
with open(exact_dir+'/XXZ4.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [77]:
observable = SparsePauliOp.from_sparse_list([("Z", [indx_qm], 1)], num_qubits=n_qubit_circuit)
print(observable)

SparsePauliOp(['IIZII'],
              coeffs=[1.+0.j])


In [78]:
def Apply_R_XXplusYYplusZZ(theta_x,theta_y,theta_z, jj, qc_: QuantumCircuit):
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(theta_z,qr[jj])
    qc_.rz(theta_x+np.pi/2,qr[jj+1])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(-theta_y,qr[jj])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.sx(qr[jj])
    qc_.sxdg(qr[jj+1])
    


In [79]:
qc = QuantumCircuit(qr_circuit)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 0, qc)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 2, qc)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 1, qc)
print(qc)

     ┌───┐┌───────┐              ┌───┐┌────────┐┌───┐ ┌────┐               »
q_0: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├───────────────»
     └─┬─┘└─┬───┬─┘┌────────────┐└─┬─┘└─┬───┬──┘└─┬─┘┌┴────┴┐┌───┐┌───────┐»
q_1: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├┤ X ├┤ Rz(3) ├»
            └───┘  └────────────┘       └───┘        └──────┘└─┬─┘└───────┘»
q_2: ──────────────────────────────────────────────────────────┼───────────»
     ┌───┐┌───────┐              ┌───┐┌────────┐┌───┐ ┌────┐   │    ┌───┐  »
q_3: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├───■────┤ H ├──»
     └─┬─┘└─┬───┬─┘┌────────────┐└─┬─┘└─┬───┬──┘└─┬─┘┌┴────┴┐       └───┘  »
q_4: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├──────────────»
            └───┘  └────────────┘       └───┘        └──────┘              »
«                                               
«q_0: ──────────────────────────────────────────
«                   ┌───┐┌────────┐┌───┐ ┌────┐ 
«q_1: 

In [80]:
def ApplyConsecutiveTrotterEvolution(times, alphas, n_trotters, indx, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    if (indx==0):
        # inter-dimer evolution first
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[0]/2.0,
                                   thetas_yy_inter[0]/2.0,
                                   thetas_zz_inter[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_inter[i_time]+thetas_xx_inter[i_time+1])/2.0,
                                       (thetas_yy_inter[i_time]+thetas_yy_inter[i_time+1])/2.0,
                                       (thetas_zz_inter[i_time]+thetas_zz_inter[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_inter[-1])/2.0,
                                   (thetas_yy_inter[-1])/2.0,
                                   (thetas_zz_inter[-1])/2.0,
                                   i_qubit, qc_)
    elif (indx==1):
        # intra-dimer evolution first
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                                   thetas_yy_intra[0]/2.0,
                                   thetas_zz_intra[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                       (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                       (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                                   (thetas_yy_intra[-1])/2.0,
                                   (thetas_zz_intra[-1])/2.0,
                                   i_qubit, qc_)

In [81]:
# test
import copy
from scipy import sparse
from datetime import datetime
def generate_numbers_with_ones(n, k):
    if k > n:
        return []
    if k==n:
        return [(1<<n-1)]
    if k==0:
        return [0]
    # Start with the lowest number with exactly k ones (e.g., for n=5, k=2: "00011")
    number = (1 << k) - 1  # All 1s
    m = n-k

    results = []

    # The highest number with n-m ones and m zeros (e.g., for n=5, m=2: "11100")
    limit = (1 << n) - (1 << m)

    while number <= limit:
        # Count 0s to ensure it has exactly m zeros
        binary_str = f'{number:0{n}b}'
        # Check if the binary representation of the number has exactly m zeros
        if binary_str.count('0') == m:
            results.append(number)
        
        # Gosper's hack to get the next combination of (n-m) 1s in an n-bit number
        c = number & -number
        r = number + c
        number = (((r ^ number) >> 2) // c) | r

    return results

# sectorize
nsec = n_qubit//2+1
dim_sub = [0 for _ in range(nsec)]
indx_sub = [[] for _ in range(nsec)]
iindx_sub = [[] for _ in range(nsec)]

Z_target = [0 for _ in range(nsec)]

for isec in range(nsec):
    Z_target[isec] = (n_qubit-2*isec)

for isec in range(nsec):
    dim_sub[isec] = 0
    indx_sub[isec] = generate_numbers_with_ones(n_qubit,isec)
    dim_sub[isec] = len(indx_sub[isec])
    
    #print(indx_sub[isec])


    if (core==0):
        st = '# dimension of subspace with Z= {Z_target}: {dim_sub}'.format(Z_target=Z_target[isec],dim_sub=dim_sub[isec])
        print(st)
    indx_sub[isec] = np.array(indx_sub[isec])
    # inverse of indx_sub
    #print(indx_sub[isec])
    iindx_sub[isec] = -np.ones((dim),dtype=int)
    for i in range(dim_sub[isec]):
        iindx_sub[isec][indx_sub[isec][i]] = i
# exact eigenvalues
eigen_energies_exact   = []
eigen_vectors_exact   = []
for isec in range(nsec):
    eigen_energies_exact.append(np.zeros((n_inter,dim_sub[isec]),dtype=float))
    eigen_vectors_exact.append(np.zeros((n_inter,dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    for alpha in range(n_hamiltonians):
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        if (np.abs(Z_target[isec])<1e-10):
            print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_exact[isec][alpha,:]   = eigen_e
        eigen_vectors_exact[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)

def ExactGaussian (isec, alpha, eps, beta):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=float)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-0.5 * beta ** 2 * (eigen_energies_exact[isec][alpha,k]-eps)**2)
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def ExactEvolution (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_exact[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol
# intra-dimer terms
hamiltonian_list = []
for i in range(0,n_qubit,2):
    ii = i
    #jj = (i+1)%n_qubit # periodic boundary condition
    jj = (i+1) # open boundary condition
    if (jj>=n_qubit):
        continue
    term = ('XX',[ii,jj],-J/4)
    hamiltonian_list.append(term)
    term = ('YY',[ii,jj],-J/4)
    hamiltonian_list.append(term)
    term = ('ZZ',[ii,jj],-J*Delta/4)
    hamiltonian_list.append(term)
    term = ('Z',[ii],-h/2)
    hamiltonian_list.append(term)
    term = ('Z',[jj],-h/2)
    hamiltonian_list.append(term)
# inter-dimer terms
hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
hamiltonian_intra = hamiltonian.simplify()
if (core==0):
    print(single_line(str(hamiltonian_intra)))
    print('')

hamiltonians_inter       = []
for alpha in range(n_hamiltonians):
    hamiltonian_list = []
    hamiltonians_inter.append((hamiltonians[alpha]-hamiltonian_intra).simplify())

    if (core==0):
        print(alpha, single_line(str(hamiltonians_inter[alpha])))
        print('')
# exact eigenvalues for intra-dimer parts
eigen_energies_intra   = []
eigen_vectors_intra  = []
for isec in range(nsec):
    eigen_energies_intra.append(np.zeros((dim_sub[isec]),dtype=float))
    eigen_vectors_intra.append(np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    start_time = datetime.now()
    # project hamiltonian on to specified sector
    H_sparse = hamiltonian_intra.to_matrix(sparse=True)
    H_sparse.eliminate_zeros()
    jsec = isec
    row      = []
    col      = []
    data     = []
    for ii in range(dim_sub[isec]):
        i = indx_sub[isec][ii]
        for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
            # j is always in indx_sub[isec], because the Hamiltonian does not mix it
            #print(i,j)
            j = H_sparse.indices[ind]
            jj = iindx_sub[jsec][j]
            row.append(jj)
            col.append(ii)
            #print(ii,jj)
            data.append(H_sparse.data[ind])
    H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

    # diagonalize sectorized hamiltonian
    eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
    #if (isec==nsec-1):
    if (np.abs(Z_target[isec])<1e-10):
        print(eigen_e[0],eigen_e[1] )
    #    if (alpha==6):
    #        for k in range(dim_sub[isec]):
    #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
    #            print(k,np.abs(overlap)**2,eigen_e[k])
    #
    eigen_energies_intra[isec][:]   = eigen_e
    eigen_vectors_intra[isec][:,:] = eigen_v
    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    #if (core==0):
    #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
    #    memory_usage(st)
# exact eigenvalues for inter-dimer parts
eigen_energies_inter   = []
eigen_vectors_inter  = []
for isec in range(nsec):
    eigen_energies_inter.append(np.zeros((n_inter,dim_sub[isec]),dtype=float))
    eigen_vectors_inter.append(np.zeros((n_inter,dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    for alpha in range(n_hamiltonians):
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians_inter[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        if (np.abs(Z_target[isec])<1e-10):
            print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_inter[isec][alpha,:]   = eigen_e
        eigen_vectors_inter[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)

def ExactEvolution_intra (isec, eps, time):
    Vl = copy.deepcopy(eigen_vectors_intra[isec][:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_intra[isec][k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def ExactEvolution_inter (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_inter[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_inter[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def TrotterEvolution(isec, alpha, time, eps, n_trotter, indx):
    dt = time/n_trotter
    if (indx==0): 
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt/2.0)
        u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
        for i_trotter in range(n_trotter-1):
            u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt/2.0)@u_trotter
    elif (indx==1):
        u_trotter = ExactEvolution_intra (isec, 0.0, dt/2.0)
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
        for i_trotter in range(n_trotter-1):
            u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
        u_trotter = ExactEvolution_intra (isec, 0.0, dt/2.0)@u_trotter
    return u_trotter*np.exp(1j*eps*time)

n_trotter = 2

norms_approximate  = np.ones((nsec,n_hamiltonians),dtype=float)
#for isec in range(nsec):
for isec in range(nsec-1,nsec):
    # 
    H_subs = [0 for _ in range(n_hamiltonians)]
    for alpha in range(n_hamiltonians):
        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))
        H_subs[alpha] = H_sub.toarray()


    phi = eigen_vectors_exact[isec][0,:,0]
    E_approximate = eigen_energies_exact[isec][0,0]
    eps = np.real(phi.conj()@H_subs[1]@phi)
    for alpha in range(1,n_hamiltonians):
        # norm calculation (i_obs = 0)
        i_obs = 0
        norm = 0.0
        for imc in range(nmc):
            times = O_timelists[alpha][i_obs][imc]
            indx = 0
            indx_dag = 0 # same for second order trotter
            evol_norm = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_norm
                i_time += 1
            evol_norm = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_norm
            i_time += 1
            #evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_norm
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_norm
                i_time += 1
            norm += phi.conj()@evol_norm@phi

            indx = 1
            indx_dag = 1
            evol_norm = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_norm
                i_time += 1
            evol_norm = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_norm
            i_time += 1
            #evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_norm
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_norm
                i_time += 1
            norm += phi.conj()@evol_norm@phi

        norm /= (2*nmc)
        norm = norm.real
        print(alpha,norm)

        # dE1 calculation (i_obs = 1)
        i_obs = 1
        dE = 0.0
        for imc in range(nmc):
            times = O_timelists[alpha][i_obs][imc]
            indx = 0
            indx_dag = 0
            evol_dE = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_dE
                i_time += 1
            evol_dE = (H_subs[alpha]-H_subs[alpha-1])@evol_dE
            evol_dE = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_dE
            i_time += 1
            #evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_dE
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_dE
                i_time += 1
            dE += phi.conj()@evol_dE@phi

            indx = 1
            indx_dag = 1
            evol_dE = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_dE
                i_time += 1
            evol_dE = (H_subs[alpha]-H_subs[alpha-1])@evol_dE
            evol_dE = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_dE
            i_time += 1
            #evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_dE
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_dE
                i_time += 1
            dE += phi.conj()@evol_dE@phi

        dE /= (2*nmc)
        dE /= norm
        dE = dE.real
        E_approximate += dE
        print(alpha,E_approximate-eigen_energies_exact[isec][alpha,0])
        print(E_approximate, eigen_energies_exact[isec][0,0], eps)
        #if (core==0):
        #    print(alpha, norms_approximate[isec,alpha])



# dimension of subspace with Z= 4: 1
# dimension of subspace with Z= 2: 4
# dimension of subspace with Z= 0: 6
0 -1.5 -0.5000000000000001
1 -1.6160254037844386 -0.9571067811865472
SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

0 SparsePauliOp(['IIII'],              coeffs=[0.+0.j])

1 SparsePauliOp(['IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j])



-1.5 -0.5000000000000001
0 0.0 0.0
1 -0.75 -0.75
1 0.9203552614850566
1 -0.0002204552024744899
-1.616245858986913 -1.5 -1.5000000000000004


In [82]:
from qiskit.quantum_info import Operator
qc_ref = QuantumCircuit(qr_circuit)
qc_ref.rxx(1.0,qr[0],qr[1])
qc_ref.ryy(2.0,qr[0],qr[1])
qc_ref.rzz(3.0,qr[0],qr[1])

u_ref = Operator(qc_ref).data
#print(qc_ref.decompose())
#print(u_ref)

qc = QuantumCircuit(qr_circuit)
Apply_R_XXplusYYplusZZ(1.0,2.0,3.0,0,qc)
print(qc)

u = Operator(qc).data
phase = np.pi/4.0 # = np.angle(u_ref[0,0]/u[0,0])
#print('')
print(np.max(np.abs(u*np.exp(1j*phase)-u_ref)))

     ┌───┐┌───────┐              ┌───┐┌────────┐┌───┐ ┌────┐ 
q_0: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├─
     └─┬─┘└─┬───┬─┘┌────────────┐└─┬─┘└─┬───┬──┘└─┬─┘┌┴────┴┐
q_1: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├
            └───┘  └────────────┘       └───┘        └──────┘
q_2: ────────────────────────────────────────────────────────
                                                             
q_3: ────────────────────────────────────────────────────────
                                                             
q_4: ────────────────────────────────────────────────────────
                                                             
2.0407899217551515e-16


In [83]:
# Single trotter test
from qiskit.quantum_info import Operator
#times = [2.0,3.0]
#n_trotters = [2,2]
#alphas = [1,2]
times = [2.0]
n_trotters = [2]
alphas = [1]
indx = 1
qc = QuantumCircuit(qr_circuit)
qc.h(qm)
for inst in init_circuit.data:
    qc.append(inst.operation, inst.qubits, inst.clbits)
ApplyConsecutiveTrotterEvolution(times,alphas,n_trotters,indx,qc)
for inst in init_circuit_inv.data:
    qc.append(inst.operation, inst.qubits, inst.clbits)
qc.h(qm)

unitary = Operator(qc).data
print(np.abs(unitary[0,0]))
Z_mat = observable.to_matrix()
evol_real = np.real((unitary.conj().T@Z_mat@unitary)[0,0])
print(evol_real)
#print(qc)

qc = QuantumCircuit(qr_circuit)
qc.h(qm)
for inst in init_circuit.data:
    qc.append(inst.operation, inst.qubits, inst.clbits)
ApplyConsecutiveTrotterEvolution(times,alphas,n_trotters,indx,qc)
for inst in init_circuit_inv.data:
    qc.append(inst.operation, inst.qubits, inst.clbits)
qc.sdg(qm)
qc.h(qm)

unitary = Operator(qc).data
Z_mat = observable.to_matrix()
evol_imag = np.real((unitary.conj().T@Z_mat@unitary)[0,0])
print(evol_imag)

evol = evol_real + 1j * evol_imag
print(evol)

# exact value
eps = 0.0
isec = nsec-1
phi = eigen_vectors_exact[isec][0,:,0]
#print(phi)
u_exact = ExactEvolution(isec, alphas[0], eps, times[0])
#u_exact = ExactEvolution(isec, alphas[1], eps, times[1])@u_exact
u_trotter = TrotterEvolution(isec,alphas[0],times[0],eps,n_trotters[0],indx)
#u_trotter = TrotterEvolution(isec,alphas[1],times[1],eps,n_trotters[1],indx)@u_trotter
#u_trotter = ExactEvolution_inter(isec,alphas[0],eps,times[0]) # weird.. how?
evol_exact = phi.conj()@u_exact@phi
evol_trotter = phi.conj()@u_trotter@phi
print(evol_exact,evol_trotter)
print(np.abs(evol_trotter/evol))


0.6371333507117501
-0.030629098810772956
-0.8270886786096853
(-0.030629098810772956-0.8270886786096853j)
(-0.8640062082485971-0.09968887915737831j) (-0.827183427129617-0.027953566147940888j)
1.0000000000000075


In [84]:
print(qc)

     ┌───┐┌───┐┌───┐┌───┐┌─────┐┌───┐┌─────┐     ┌───┐┌──────────┐»
q_0: ┤ S ├┤ H ├┤ T ├┤ X ├┤ Tdg ├┤ H ├┤ Sdg ├──■──┤ X ├┤ Rz(0.25) ├»
     └───┘├───┤└───┘└─┬─┘└─────┘└───┘└─────┘┌─┴─┐└─┬─┘└──┬───┬───┘»
q_1: ─────┤ X ├──■────■─────────────────────┤ X ├──■─────┤ H ├────»
     ┌───┐└─┬─┘  │                          └───┘        └───┘    »
q_2: ┤ H ├──■────┼────────────────────────────────────────────────»
     └───┘     ┌─┴─┐                        ┌───┐┌───┐┌──────────┐»
q_3: ──────────┤ X ├──■─────────────────────┤ X ├┤ X ├┤ Rz(0.25) ├»
     ┌───┐┌───┐├───┤┌─┴─┐┌─────┐┌───┐┌─────┐└─┬─┘└─┬─┘└──┬───┬───┘»
q_4: ┤ S ├┤ H ├┤ T ├┤ X ├┤ Tdg ├┤ H ├┤ Sdg ├──■────■─────┤ H ├────»
     └───┘└───┘└───┘└───┘└─────┘└───┘└─────┘             └───┘    »
«                   ┌───┐┌──────────┐┌───┐ ┌────┐                 »
«q_0: ──────────────┤ X ├┤ Rz(0.25) ├┤ X ├─┤ √X ├─────────────────»
«     ┌────────────┐└─┬─┘└──┬───┬───┘└─┬─┘┌┴────┴┐┌───┐┌─────────┐»
«q_1: ┤ Rz(1.3208) ├──■─────┤ H ├──────■──┤ √Xdg

In [85]:
def NumberOfTrotterSteps(alpha):
    return 2
if (core==0):
    print('# of Trotter Steps for each alpha')
    for alpha_ in range(1,n_hamiltonians):
        print(NumberOfTrotterSteps(alpha_))

# of Trotter Steps for each alpha
2


In [86]:
from qiskit import qpy
backend_options = {'method':'statevector', 'max_parallel_threads':1}
if (core==0):
    code_run_pass_manager = '''from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import qpy
from qiskit_aer import AerSimulator
backend_options = '''+ str(backend_options) +'''
sim = AerSimulator(**backend_options)
pass_manager = generate_preset_pass_manager(2, sim)
with open('circuits.qpy', 'rb') as file_:
    circuits = qpy.load(file_)
    n_circuit = len(circuits)
    circuits_opt = []
    for i_circuit in range(n_circuit):
        circuit_opt = pass_manager.run(circuits[i_circuit])
        circuits_opt.append(circuit_opt)
with open ('circuits_opt.qpy', 'wb') as file_:
    qpy.dump(circuits_opt,file_)
    '''
    
    with open('run_pass_manager.py', 'w') as file_:
        file_.write(code_run_pass_manager)

In [87]:
# make a circuit
def ApplyControlledPauliGate_bare(paulis, qc_, qr_, qm_):
    for i in range(len(qr_)):
        p = paulis[-i-1]
        if p=="I":
            continue
        if p=="X":
            qc_.cx(qm_,qr_[i])
        if p=="Y":
            qc_.cy(qm_,qr_[i])
        if p=="Z":
            qc_.cz(qm_,qr_[i])
nhd = len(hamiltonian_diffs_reduced_list[0]) # same difference elements for all
circuits = []
for ihd in range(nhd):
    circuit = QuantumCircuit(qr_circuit)
    pauli_ = hamiltonian_diffs_reduced_list[alpha-1][ihd][0]
    for i in range(len(qr)):
        p = pauli_[-i-1]
    ApplyControlledPauliGate_bare(pauli_, circuit, qr, qm)
    circuits.append(circuit)
if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    pass_manager_run = subprocess.run(['python3', 'run_pass_manager.py'])

In [88]:
with open('circuits_opt.qpy', 'rb') as file_:
    paulis_opt = qpy.load(file_)
for circuit in paulis_opt:
    print(circuit)


               
q_0: ──────────
     ┌───┐     
q_1: ┤ X ├─────
     └─┬─┘     
q_2: ──■────■──
          ┌─┴─┐
q_3: ─────┤ X ├
          └───┘
q_4: ──────────
               
           
q_0: ──────
           
q_1: ─■────
      │    
q_2: ─■──■─
         │ 
q_3: ────■─
           
q_4: ──────
           


In [89]:
def ApplyControlledPauliGate(ihd, qc_):
    for inst in paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)
empty_paulis_opt = []
for ihd in range(nhd):
    circuit = QuantumCircuit(qr_circuit)
    empty_paulis_opt.append(circuit)
def ApplyEmptyPauliGate(ihd, qc_):
    for inst in empty_paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)

In [90]:
print(empty_paulis_opt[1])

     
q_0: 
     
q_1: 
     
q_2: 
     
q_3: 
     
q_4: 
     


In [91]:
#circuit = QuantumCircuit(qr_circuit)
#ApplyControlledPauliGate(0,circuit)
#print(circuit)
#ApplyControlledPauliGate(1,circuit)
#print(circuit)

In [92]:
run_circuits_file = 'circuits_opt.qpy'
if (core==0):
    code_run_estimator = '''from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit import qpy
import sys, pickle
from qiskit.quantum_info import SparsePauliOp


pickled_input_data = sys.stdin.buffer.read()
input_data = pickle.loads(pickled_input_data)
n_qubit ='''+ str(n_qubit) + '''
observable = SparsePauliOp.from_sparse_list([("Z", ['''+str(indx_qm)+'''], 1)], num_qubits='''+str(n_qubit_circuit)+''')
backend_options = '''+ str(backend_options) +'''
estimator = Estimator(options = {"default_precision":0.00,
                                     'backend_options':backend_options})
run_circuits_file = \'''' + run_circuits_file + '''\'
with open(run_circuits_file, 'rb') as file_:
    circuits = qpy.load(file_)
pubs = []
len_input = len(input_data)
for ind in range(len_input):
    i0     = input_data[ind][0]
    if (len(input_data[ind])>1):
        params = input_data[ind][1]
        pubs.append((circuits[i0],observable,params))
    else:
        pubs.append((circuits[i0],observable))
job = estimator.run(pubs)
result = job.result()
len_result = len(result)
list_out = []
for i in range(len_result):
    list_out.append(result[i].data.evs)
sys.stdout.buffer.write(pickle.dumps(list_out))
    '''
    
    with open('run_estimator.py', 'w') as file_:
        file_.write(code_run_estimator)

In [93]:
eigen_energies_ref = np.zeros((n_hamiltonians),dtype=float)

for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    eigen_energies_ref[i_inter] = - J/4 * Delta * (n_dimer)
    # inter-dimer energies
    eigen_energies_ref[i_inter] += - J/4 * Delta * t_inter* (n_dimer-1) # open boundary condition

In [94]:
print(eigen_energies_ref)

[0.5  0.75]


In [95]:
norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)


eigen_energies_qzmc[0]  = -0.25 * J * (2.0-Delta) * n_dimer
print(eigen_energies_qzmc[0])

-1.5


In [96]:
pauli_identity = 'I'*n_qubit

In [106]:
eps = eigen_energies_qzmc[0]
# first-order perturbation correction = 0
from qiskit.circuit import ParameterVector

#comm.Barrier()
for alpha in range(1,n_hamiltonians):
    start_time = datetime.now()
    circuits = []
    times = ParameterVector('t', 2*alpha-1)

    # dE1
    nhd = len(hamiltonian_diffs_reduced_list[alpha-1])
    for ihd in range(nhd):
        # norm for ihd
        # implement quantum circuits
        # 1, norm, indx = 0
        indx = 0
        circuit = QuantumCircuit(qr_circuit)
        circuit.h(qm)
        for inst in init_circuit.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        i_time  = 0
        phase   = 0.0
        # \mathcal{P}_{\alpha-1}
        evolution_times = []
        alphas = []
        n_trotters = []
        for alpha_ in range(1,alpha):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1

        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
        ApplyEmptyPauliGate(ihd,circuit)

        evolution_times = []
        alphas = []
        n_trotters = []

        # P_{\alpha}
        alphas.append(alpha)
        evolution_times.append(times[i_time])
        n_trotters.append(NumberOfTrotterSteps(alpha))
        #print(evolution_times,alphas,n_trotters)
        phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
        i_time += 1

        # \mathcal{P}^{\dagger}_{\alpha-1}
        for alpha_ in reversed(range(1,alpha)):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1
        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
        #print(circuit)

        for inst in init_circuit_inv.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)

        circuit.rz(phase,qm)
        circuit.h(qm)


        circuits.append(circuit)

        if (ihd==0):
            qc = circuit

        del circuit


        # 2, norm, indx = 1
        indx = 1
        circuit = QuantumCircuit(qr_circuit)
        circuit.h(qm)
        for inst in init_circuit.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        i_time  = 0
        phase   = 0.0
        # \mathcal{P}_{\alpha-1}
        evolution_times = []
        alphas = []
        n_trotters = []
        for alpha_ in range(1,alpha):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1

        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
        ApplyEmptyPauliGate(ihd,circuit)

        evolution_times = []
        alphas = []
        n_trotters = []

        # P_{\alpha}
        alphas.append(alpha)
        evolution_times.append(times[i_time])
        n_trotters.append(NumberOfTrotterSteps(alpha))
        phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
        i_time += 1

        # \mathcal{P}^{\dagger}_{\alpha-1}
        for alpha_ in reversed(range(1,alpha)):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1
        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)

        for inst in init_circuit_inv.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)

        circuit.rz(phase,qm)
        circuit.h(qm)


        circuits.append(circuit)

        del circuit


        # dE*norm for ihd
        # 1, dE*norm, indx = 0
        indx = 0
        circuit = QuantumCircuit(qr_circuit)
        circuit.h(qm)
        for inst in init_circuit.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        i_time  = 0
        phase   = 0.0
        # \mathcal{P}_{\alpha-1}
        evolution_times = []
        alphas = []
        n_trotters = []
        for alpha_ in range(1,alpha):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1

        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
        ApplyControlledPauliGate(ihd,circuit)

        evolution_times = []
        alphas = []
        n_trotters = []

        # P_{\alpha}
        alphas.append(alpha)
        evolution_times.append(times[i_time])
        n_trotters.append(NumberOfTrotterSteps(alpha))
        phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
        i_time += 1

        # \mathcal{P}^{\dagger}_{\alpha-1}
        for alpha_ in reversed(range(1,alpha)):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1
        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)

        for inst in init_circuit_inv.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)

        circuit.rz(phase,qm)
        circuit.h(qm)


        circuits.append(circuit)

        del circuit


        # 2, dE*norm, indx = 1
        indx = 1
        circuit = QuantumCircuit(qr_circuit)
        circuit.h(qm)
        for inst in init_circuit.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        i_time  = 0
        phase   = 0.0
        # \mathcal{P}_{\alpha-1}
        evolution_times = []
        alphas = []
        n_trotters = []
        for alpha_ in range(1,alpha):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1

        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
        ApplyControlledPauliGate(ihd,circuit)

        evolution_times = []
        alphas = []
        n_trotters = []

        # P_{\alpha}
        alphas.append(alpha)
        evolution_times.append(times[i_time])
        n_trotters.append(NumberOfTrotterSteps(alpha))
        phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
        i_time += 1

        # \mathcal{P}^{\dagger}_{\alpha-1}
        for alpha_ in reversed(range(1,alpha)):
            alphas.append(alpha_)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha_))
            phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
            i_time += 1
        ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)

        for inst in init_circuit_inv.data:
            circuit.append(inst.operation, inst.qubits, inst.clbits)

        circuit.rz(phase,qm)
        circuit.h(qm)


        circuits.append(circuit)

        del circuit

    if (core==0):
        with open ('circuits.qpy', 'wb') as file_:
            qpy.dump(circuits,file_)
        del circuits[:]
        subprocess.run(['python3', 'run_pass_manager.py'])
    #comm.Barrier()

    estimator_inputs = []

    n_pubs = 0
    # dE1
    n_pubs += nhd * nmc * 2 * 2 # 2 for indx, 2 for norm

    n_pubs_for_ = [0 for _ in range(cores)]
    remainder         = n_pubs%cores
    for i_core in range(cores):
        n_pubs_for_[i_core] = n_pubs//cores
        if (i_core<remainder):
            n_pubs_for_[i_core] += 1
    if (core==0 and alpha==1):
        print('# of different quantum circuits to run = ', n_pubs)

    i_start         = sum(n_pubs_for_[:core])
    i_end           = i_start + n_pubs_for_[core]
    ind_pub         = 0
    indx_circuit    = 0
    ## dE1
    for ihd in range(nhd):
        # norm, indx = 0, 1
        i_obs = 0
        for indx in range(2):
            for imc in range(nmc):
                my_turn = ind_pub>=i_start and ind_pub<i_end
                if (my_turn):
                    estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
                ind_pub += 1
            indx_circuit += 1
        # dE*norm, indx = 0, 1
        i_obs = 1
        for indx in range(2):
            for imc in range(nmc):
                my_turn = ind_pub>=i_start and ind_pub<i_end
                if (my_turn):
                    estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
                ind_pub += 1
            indx_circuit += 1

    result_values_core = [0.0 for _ in range(n_pubs_for_[core])]

    gc.collect()
    serialized_estimator_inputs = pickle.dumps(estimator_inputs)
    estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
                   stdout=subprocess.PIPE,stderr=subprocess.PIPE)
    try:
        result = pickle.loads(estimator_run.stdout)
    except EOFError: # empty list
        result = []
    gc.collect()

    for i in range(n_pubs_for_[core]):
        #print(result[i])
        computed_value = result[i]

        # # shot errors
        p_up = (computed_value + 1.0)/2.0
        sample = np.random.binomial(n_shot,p_up)
        shot_error = 2*(sample/(n_shot) - p_up)
        computed_value += shot_error

        result_values_core[i] = computed_value
    #print(core,result_values_core)
    del result

    ### bcast
    #comm.Barrier()
    result_values = []
    for i_core in range(cores):
        if (n_pubs_for_[i_core]==0):
            continue
        #result_values_temp = comm.bcast(result_values_core,root=i_core)
        result_values_temp = result_values_core
        result_values += result_values_temp
        #comm.Barrier()
    print(result_values)

    norms_1    = np.zeros((nhd,2),dtype=float)
    pauli_norms_1  = np.zeros((nhd,2),dtype=float)
    i_meas = 0
    for ihd in range(nhd):
        # norm, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
        # pauli, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                pauli_norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
    norms_1 /= nmc
    pauli_norms_1 /= nmc

    print(norms_1)
    print(pauli_norms_1)
    norm_1 = np.sum(norms_1)/(nhd*2)

    # dE1
    dE1 = 0.0
    for ihd in range(nhd):
        norm_avg = 0.0
        for indx in range(2):
            norm_avg += norms_1[ihd,indx]
        pauli_avg = 0.0
        for indx in range(2):
            pauli_avg += pauli_norms_1[ihd,indx]
        pauli_avg /= norm_avg
        coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
        dE1 += coeff * pauli_avg
    dE1 = dE1.real
    
    eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
    norms_qzmc[alpha] = norm_1
    del times
    del result_values[:]
    del result_values_core[:]
    gc.collect()

    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    if (core==0):
        st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
        memory_usage(st)
        print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1][alpha,0])
        if (alpha<n_hamiltonians-1):
            print('precision of the predictor for next', eps-eigen_energies_exact[-1][alpha+1,0])
        st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
        print(st)



# of different quantum circuits to run =  2400
[array(0.988), array(0.9915), array(0.8585), array(0.9245), array(0.911), array(0.9725), array(0.9645), array(0.9025), array(0.9165), array(0.9955), array(0.9335), array(0.9165), array(0.824), array(1.), array(0.9825), array(0.918), array(0.943), array(0.994), array(0.8985), array(0.9475), array(0.9685), array(0.9335), array(0.9125), array(0.997), array(0.9995), array(0.9545), array(0.996), array(0.927), array(0.959), array(0.9025), array(0.996), array(0.9965), array(0.9335), array(0.9985), array(0.9905), array(0.9005), array(0.983), array(0.9), array(0.9825), array(0.9965), array(0.999), array(0.91), array(0.9215), array(0.9975), array(0.942), array(0.991), array(0.9615), array(0.9125), array(0.956), array(0.982), array(0.964), array(0.9635), array(0.9585), array(0.9985), array(0.996), array(0.9965), array(0.972), array(0.915), array(0.9165), array(0.9195), array(-0.168), array(0.9165), array(0.974), array(0.9275), array(0.997), array(0.9

In [98]:
print(O_timelists[1][0][0][0])

-0.37029690306220664


In [99]:
qc_ = qc.assign_parameters([O_timelists[1][0][0][0]])

In [100]:
print(observable)

SparsePauliOp(['IIZII'],
              coeffs=[1.+0.j])


In [101]:
unitary = Operator(qc_).data
Z_mat = observable.to_matrix()
evol = np.real((unitary.conj().T@Z_mat@unitary)[0,0])
print(evol)

0.9877472936185617
